# ED5J990H5VAZT

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import math
import os
import gc
from pathlib import Path
import re
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Set directory to project root
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Custom packages
from tools.filter import FilterDF as fdf
from tools.benchmarks import ParetoAnalysis as pa
from tools.benchmarks import AccuracyCalculation as ac
from tools.integrity_fixes import DataFixer as fix, DataExporter as exporter

# Preemptively set new Pandas option, also set matplotlib to close
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

# Allow reloading of custom Python classes without resetting kernel
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Read data from parquet files

In [ ]:
# Load formatted data
%store -r static_data_merged
%store -r sales_data_merged

# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    
    %store static_data_merged
    
# Data already exists
else:
    static_data = static_data_merged.copy()

# 

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
        
    %store sales_data_merged
    
# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()

# 

# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# True promos
%store -r before_after_details_true
if 'before_after_details_true' not in locals():
    before_after_details_true = pd.read_csv('data/before_after_details_true.csv', index_col='location_id')
    %store before_after_details_true

# Timezones
%store -r timezones
if 'timezones' not in locals():
    timezones = pd.read_csv('data/timezones.csv', index_col='location_id')['timezone'].to_dict()
    for loc_id, df in sales_and_menu_data.items():
        df.index = df.index.tz_convert(timezones[loc_id])
        sales_and_menu_data[loc_id] = df
    %store timezones

%store -r restaurants_by_4m_coverage
if 'restaurants_by_4m_coverage' not in locals():
    restaurants_by_4m_coverage = pd.read_csv('data/2_palate_data_parquet_cleaned/restaurants_by_4m_coverage.csv')['location_id'].tolist()
    %store restaurants_by_4m_coverage

loc_id = 'ED5J990H5VAZT'
df_uncleaned = sales_and_menu_data[loc_id]

locations = list(sales_and_menu_data.keys())
for other_loc_id in locations:
    if other_loc_id != loc_id:
        del sales_and_menu_data[other_loc_id]
        del sales_data_merged[other_loc_id]
gc.collect()

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Vegan") or item_modifications.str.contains("Vegan")').query('item_name.str.contains("Bacon") or item_modifications.str.contains("Bacon")')['item_quantity'].sum()

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
df_uncleaned.query('item_name.str.contains("Vegan") or item_modifications.str.contains("Vegan")').query('item_name.str.contains("Bacon") or item_modifications.str.contains("Bacon")')['item_quantity'].sum()

In [ ]:
# original_query = pd.DataFrame.query
# pd.DataFrame.query = partialmethod(pd.DataFrame.query, engine="python")

In [ ]:
df['item_name'].value_counts().size

### Diagnostics

In [ ]:
sales_and_menu_data[loc_id].index

In [ ]:
time_differences_details[loc_id]

### Dish Consolidation

In [ ]:
loc_id

In [ ]:
print(df_uncleaned['item_name'].value_counts().to_string())

In [ ]:
df_uncleaned.head(1)

In [ ]:
df_uncleaned['dish_category'].value_counts()

In [ ]:
drink_categories = ["Coffee & Tea","Dairy Drink","Alcohol","Soda","Water","Juice","Sports & Health Drink"]

In [ ]:
print(df_uncleaned.query('(is_plant_based == "Yes")')['item_name'].value_counts().to_string())

In [ ]:
df_uncleaned.query('item_name == "Matcha"')['dish_category'].value_counts()

In [ ]:
df_uncleaned.query('~dish_category.isin(@drink_categories)')['dish_category'].value_counts()

In [ ]:
print(df_uncleaned.query('(is_plant_based == "Yes") and ~dish_category.isin(@drink_categories)')['item_name'].value_counts().to_string())

In [ ]:
df_uncleaned.query('item_name == "Veggie Sandwich With Side Salad"')['item_modifications'].value_counts()

In [ ]:
# Rename items based on modications
modification_name_changes = [
    (('Veggie Sandwich With Side Salad', 'Vegan Bacon|Field Roast Sausage|Thrilling Foods Bacon|Vegan Good Planet Cream Cheese', 'Vegan Veggie Sandwich'),('S')),
    (('Veggie Sandwich With Side Salad', 'Bacon|Sausage|Turkey', 'Meat Veggie Sandwich'),('C')),
    (('Veggie Sandwich With Side Salad', 'Cheddar|Cheese|Egg', 'Cheese Veggie Sandwich')('S')),
    ('Vegan Veggie Sandwich', '', 'Veggie Sandwich With Side Salad')]

# Item names to consolidate
dish_names = {"Griffith Street" : ["Gs", "Gs Meat"], # Vegan Griffith Street 
              "Awkward Aardvark" : ["Aa/No Meat"], 
              "Egg & Cheddar" : ["Ec", "Ec Tomato", "Egg Cheddar", "Egg Cheese"],  # "Vegan Egg & Cheddar", "Vegan Egg & Cheese"
              "Egg Hummus Pesto" : ["Ehp", "Ehp Tomato", "Ehp Meat", "Egg Hummus-Pesto"], # "Vegan Egg Hummus Pesto"
              "Egg Meat Cheese" : ["Ec Meat", "3. Egg Meat Cheese"],
              "Egg Meat" : ["Egg & Bacon"],
              "Egg": ["Scrambled Eggs"], # "Egg Sandwich"
              "Loose-Leaf Tea" : ["Loose-Leaf", "Loose Leaf"]}

# modification_names = {"12oz" : ["12oz  12oz"],
#                       "16oz" : ["16oz  16oz"],
#                       "20oz" : ["20oz  20oz"],
#                       "24oz" : ["24oz  24oz"],
#                       "Regular Price" : ["Regular"]}

others = {
              "Bagel", 
              "The Craven", 
              "Yeti Sandwich",
              "Fullmetal Alchemist",
              "Anam A Nom",
              "The Jesse Sandwich"}

# Swap the keys and values
replacement_dict = {variant: canonical for canonical, variants in dish_names.items() for variant in variants}
# replacement_dict = {variant: canonical for canonical, variants in dish_names.items() for variant in variants}

# Items to remove
items_to_remove = ["Bookmark", 
                   "Necklace", 
                   "Gift", 
                   "Card ",
                   "Postcard", # exact
                   "Egift Card", # exact
                   "Art", # exact
                   "Earring",
                   "Stickers", # exact
                   "Holographic Sticker", # exact
                   "Mask",
                   "Print", # exact
                   "Bath Bomb",
                   "Graph",
                   "Journal",
                   "Canister",
                   "Bill", # exact,
                   "Spatula",
                   "Towel",
                   "Pin ",
                   "Shirt"
                   ]

drinks = ["Matcha"]

# What is:
# Sandwich - Breakfast
# Flavor
# Ex. Flavor
# Kind
# Wow
# Pure

# Probably change
# Yeli Sandwich to Yeti Sandwich

In [ ]:
%store -r labels_ED5J990H5VAZT

In [ ]:
cleaned_labels = (labels_ED5J990H5VAZT
                  .query('label == "merchandise"')
                  .assign(item=lambda df: df['item']
                          .str.strip('123456789./\\ ')  # Clean up item names
                          .replace(replacement_dict)    # Replace names based on dictionary
                          .replace(items_to_remove, pd.NA))  # Replace non-dish items with NA
                  .dropna(subset=['item'])
                  .drop_duplicates('item')
                  )

In [ ]:
print(cleaned_labels.query('score < 0.7').to_string())
#[Garden Home]
#[Signature, Bear Hug, Steamer, Goblin King, French Press, Golden Fire, Hazelhoff, Mr. Tanuki, Golden Yogi, Dad'S House]

In [ ]:
df_cleaned = (df
              .assign(item_name=lambda df: df['item_name']
                      .str.strip('123456789./\\ ')  # Clean up item names
                      .replace(replacement_dict)    # Replace names based on dictionary
                      .replace(items_to_remove, pd.NA))  # Replace non-dish items with NA
              .dropna(subset=['item_name'])
              .drop('unique_id', axis=1)
             )

df = df_cleaned

In [ ]:
df.query('item_name.str.contains("Meat")')['item_name'].unique()

In [ ]:
print(df['item_name'].value_counts().to_string())

In [ ]:
# 'item_modifications.str.contains("Vegan")'
bacon_conditions = ['item_modifications.str.contains("Bacon")',
                    'item_name.str.contains("Bacon")'
                    'item_name.str.contains("Anam A Nom")',
                    'item_name.str.contains("Yeti Sandwich")',
                    'item_name.str.contains("Fullmetal Alchemist")',
                    'item_name.str.contains("The Jesse Sandwich")',]
df.query(' or '.join(bacon_conditions)).head()

In [ ]:
"Item :" + df[['item_name','item_modifications']].value_counts().reset_index()['item_name'] + "; Modifications: " + df[['item_name','item_modifications']].value_counts().reset_index()['item_modifications']

In [ ]:
items_ED5J990H5VAZT = df['item_name'].value_counts().index.tolist()
%store items_ED5J990H5VAZT

In [ ]:
%store -r time_differences
%store -r time_differences_details
time_difference_details = time_differences_details[loc_id].rename_axis(['day', 'date', 'datetime']).to_frame('gap').reset_index().sort_values(['datetime'])
%store -r before_after_details_true

In [ ]:
total_items = time_differences[loc_id].sum().sum()
colorbar_max = total_items / 10**3

plot_time_spacing(loc_id, time_differences, colorbar_max)

In [ ]:
if loc_id == 'ED5J990H5VAZT':
    promo_item_containing = df[(df['item_name'].str.contains(before_after_details_true.loc[loc_id,'promo_name'][0]) & 
                                                        df['item_name'].str.contains(before_after_details_true.loc[loc_id,'promo_name'][1]) | 
                                                        df['item_modifications'].str.contains(before_after_details_true.loc[loc_id,'promo_name'][0]) &
                                                        df['item_modifications'].str.contains(before_after_details_true.loc[loc_id,'promo_name'][1]))]

print(promo_item_containing['item_name'].unique().tolist())

plt.plot(promo_item_containing.resample('W')['item_quantity'].sum())
plt.axvline(x=before_after_details_true.loc[loc_id, 'cross_over_date'], color='red', linestyle='--', label='Promo Date')
plt.title("Plant-Based Analog")
plt.xticks(rotation=70)
plt.show()

In [ ]:
plot_time_series(loc_id, df, before_after_details_true, 1600, freq='D')

In [ ]:
gap_8 = time_difference_details.query('gap == 9')
example_datetime = pd.Timestamp('2023-02-25 08:11:31-08:00')
gap_8

In [ ]:
pd.concat([df[:example_datetime].tail(3), df[example_datetime:].head(5)])

In [ ]:
dish_conditions_egg_meat = [df['item_name'].str.contains(dish) for dish in ['Egg Meat', 'Egg Meat Cheese', 'Ex Meat']]
modification_meat_conditions = [df['item_modifications'].str.contains(meat) for meat in ['Sausage', 'Bacon', 'Ham']]
print(df
      .loc[reduce(np.logical_or, dish_conditions_egg_meat) & ~reduce(np.logical_or, modification_meat_conditions)]
      .value_counts(['item_name','item_modifications'], sort=False)
      .to_string())

In [ ]:
pd.DataFrame.query = original_query

In [ ]:
# Visualizing with gaps for inactive weeks
introduction_fig, ax = plt.subplots(figsize=(14, 8))

# Index into the promotional items for this restaurant
promo_datetime = before_after_details.loc[loc_id,'cross_over_date'].tz_convert('UTC')

for dish in food_df['item_name'].value_counts().to_frame(name='c').index[:50][::-1]:
    dish_df = food_df.query('item_name == @dish')
    dish_activity = (dish_df['item_quantity']
                                    .resample('W')
                                    .sum()
                                    .to_frame(name='W')
                                    .query('0 < W')
                                    .index
                                    .tz_localize(None)
                                    .to_period('W')
                                    .tolist())

    # For every active week
    for week in dish_activity:

        # Place a blue dot
        ax.hlines(y=dish, xmin=week.start_time, xmax=week.end_time, colors='blue', lw=2, label=loc_id)

# Place a red circle for the promotional item
ax.plot(promo_datetime, loc_id, 'ro', alpha=0.5)

# Plot
ax.set_title('Introduction Weekly Activity for Each Dish')
ax.set_xlabel('Date')
ax.set_ylabel('Dish')

# Figure
introduction_fig.tight_layout()

plt.show()